<h2 style="margin:3px;padding:3px;">ConceptTracer: Interactive Analysis of Concept Saliency and Selectivity in Neural Representations</h2>

In [ ]:
import numpy as np
import pandas as pd
import random
import torch

from concept_tracer.config import Config
from concept_tracer import helpers


cfg = Config()
random.seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
torch.manual_seed(cfg.random_seed)

tasks = cfg.dataset_specs[cfg.dataset_name].get("tasks", []) or []
# selection of first task by default if applicable, but can be changed
task = tasks[0] if tasks else None

# build interpretability df
df_list_interpret = []
for granularity in cfg.dataset_specs[cfg.dataset_name].get("granularities", {}) or [None]:
    df = pd.read_csv(
        cfg.interpret_path.format(**helpers.resolve_suffixes(cfg, task=task, granularity=granularity)),
        usecols=["neuron", "layer", "concept", "saliency", "selectivity", "p_saliency", "p_selectivity", "p_combined"]
    )
    df["granularity"] = granularity
    df_list_interpret.append(df)
df_combined_interpret = pd.concat(df_list_interpret, ignore_index=True)

# build baseline df
df_list_bl = []
for granularity in cfg.dataset_specs[cfg.dataset_name].get("granularities", {}) or [None]:
    df = pd.read_csv(
        cfg.baseline_path.format(**helpers.resolve_suffixes(cfg, task=task, granularity=granularity))
    )
    df["granularity"] = granularity
    df_list_bl.append(df)
df_combined_bl = pd.concat(df_list_bl, ignore_index=True)

# merge indices and evaluation metrics into baseline df
df_combined_baseline = df_combined_bl.merge(
    df_combined_interpret.reset_index()[["index", "neuron", "layer", "concept", "granularity",
                                         "saliency", "selectivity", "p_saliency", "p_selectivity", "p_combined"]],
    on=["neuron", "layer", "concept", "granularity"]
)
df_combined_baseline = df_combined_baseline.set_index("index")

In [2]:
# knee point calculations
knee_point_interpret = helpers.get_knee_point(
    df_combined_interpret
)
knee_point_shap = helpers.get_knee_point(
    df_combined_baseline[df_combined_baseline["baseline"] == "shap"]
)
knee_point_l0l2 = helpers.get_knee_point(
    df_combined_baseline[df_combined_baseline["baseline"] == "l0l2"]
)

# Pareto front calculations
pareto_front_interpret = helpers.get_pareto_front(
    df_combined_interpret
)
pareto_front_shap = helpers.get_pareto_front(
    df_combined_baseline[df_combined_baseline["baseline"] == "shap"]
)
pareto_front_l0l2 = helpers.get_pareto_front(
    df_combined_baseline[df_combined_baseline["baseline"] == "l0l2"]
)

In [ ]:
fig = helpers.get_scatter_plot_publication(
    # exclude insignificant values for plotting for improved readability
    df_combined_interpret[
        (df_combined_interpret["p_selectivity"] < cfg.significance_threshold) &
        (df_combined_interpret["p_saliency"] < cfg.significance_threshold) &
        (df_combined_interpret["p_combined"] < cfg.significance_threshold)
    ], {
        "interpret_front": pareto_front_interpret,
        "interpret_knee": knee_point_interpret,
        "shap_front": pareto_front_shap,
        "shap_knee": knee_point_shap,
        "l0l2_front": pareto_front_l0l2,
        "l0l2_knee": knee_point_l0l2
    }
)
fig.write_image(f"plots/scatter_{task}.pdf")
fig.write_image(f"plots/scatter_{task}.svg")

In [ ]:
print(helpers.get_top_k_features(
    task, df_combined_interpret, pareto_front_interpret[8], cfg, 4, "high_level"
))

['age' 'triage_sbp' 'eci_Arrhythmia' 'eci_HTN2']


In [5]:
print(df_combined_interpret.loc[pareto_front_interpret[8]])

concept                   I
layer                    22
neuron                  145
saliency           0.119927
selectivity        0.305015
p_saliency         0.009901
p_selectivity      0.009901
p_combined         0.019802
granularity      high_level
Name: 41233, dtype: object
